# G1.1 — Step response and time constant (guided)

Exercise card: `G1.1_step_response.md` · 15 min · prerequisites: lesson §5, notebook
`s01_thermal_zone.ipynb` §2.

**Goal.** Read the time constant τ and the steady state $T_{ss}$ from a simulated step response of
the thermal zone (M01) and relate them to $C$ and $UA$:

$$C\,\frac{dT}{dt} = \dot Q - UA\,(T - T_{amb})$$

**How to work.** Run the cells from the top (*Shift+Enter*). Complete the cells marked **TODO** and write your answers in the cells marked *Your answer*. Before the self-check, run **Kernel → Restart Kernel and Run All Cells…**: the notebook must run from top to bottom.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from energy_training.plotting.style import apply_style, degc, hours
from energy_training.simulation.integrators import explicit_euler
from energy_training.utilities import celsius_to_kelvin

apply_style()
H = 3600.0  # s per hour

# Parameters (illustrative), SI units
C = 8.0e6  # J/K
UA = 250.0  # W/K
T_AMB = float(celsius_to_kelvin(-5.0))  # K
Q = 6000.0  # W, constant heater power
T0 = float(celsius_to_kelvin(12.0))  # K, initial zone temperature

## 1. Predict by hand

Before writing any code, compute $\tau = C/UA$ (in hours) and $T_{ss} = T_{amb} + \dot Q/UA$
(in °C).

*Your answer:* τ = … h, $T_{ss}$ = … °C

## 2. The model equation

Complete `rhs(t, y)`: it returns $dT/dt$ in K/s as a one-element array, from the energy balance
above. `y[0]` is the zone temperature in K.

In [ ]:
def rhs(t, y):
    """Return dT/dt [K/s] as a 1-element array. y[0] is the zone temperature [K]."""
    # TODO: energy balance C dT/dt = Q - UA (T - T_amb)
    raise NotImplementedError("TODO: complete rhs()")

## 3. Simulate 5 days

Explicit Euler with step $h$ = 10 min (provided).

In [ ]:
h = 600.0  # s
t, y = explicit_euler(rhs, (0.0, 5 * 24 * H), T0, h)
T = y[:, 0]

fig, ax = plt.subplots()
ax.plot(hours(t), degc(T), label="Euler, h = 10 min")
ax.set(xlabel="time [h]", ylabel="T [°C]")
ax.legend();

## 4. Read τ and $T_{ss}$ from the simulation

$T_{ss}$ is estimated as the final value. Complete `estimate_tau`: the time at which the
temperature has covered 63.2 % ($1 - e^{-1}$) of the step from $T_0$ to $T_{ss}$. Interpolate
between the two samples around that level (`np.interp`).

In [ ]:
def estimate_tau(t, T, T0, T_ss):
    """Return the time [s] at which 63.2 % of the step T0 -> T_ss is reached (interpolate)."""
    # TODO
    raise NotImplementedError("TODO: complete estimate_tau()")

In [ ]:
T_ss_est = float(T[-1])
tau_est = estimate_tau(t, T, T0, T_ss_est)
print(f"T_ss estimate: {degc(T_ss_est):.3f} degC")
print(f"tau estimate:  {tau_est / H:.3f} h")

fig, ax = plt.subplots()
ax.plot(hours(t), degc(T))
ax.axhline(degc(T_ss_est), ls="--", lw=1, color="k", label="$T_{ss}$ estimate")
level = T0 + (1 - np.exp(-1)) * (T_ss_est - T0)
ax.axhline(degc(level), ls=":", lw=1, color="gray", label="63.2 % of the step")
ax.axvline(tau_est / H, ls=":", lw=1, color="k", label="τ estimate")
ax.set(xlabel="time [h]", ylabel="T [°C]", xlim=(0, 48))
ax.legend();

## Self-check

Compares your estimates with the expected outcome of the card: τ within 2 % of $C/UA$,
$T_{ss}$ within 0.05 K of $T_{amb} + \dot Q/UA$.

In [ ]:
TAU = C / UA
T_SS = T_AMB + Q / UA
print(f"tau:  {tau_est / H:.3f} h  vs C/UA = {TAU / H:.3f} h")
print(f"T_ss: {degc(T_ss_est):.3f} degC vs {degc(T_SS):.3f} degC")
assert abs(tau_est - TAU) / TAU < 0.02, "tau is more than 2 % from C/UA: check rhs and estimate_tau"
assert abs(T_ss_est - T_SS) < 0.05, "T_ss is more than 0.05 K off: check rhs"
print("Self-check passed.")

## 5. Which estimate depends on the simulation length?

Compare both estimates with your prediction. Which of the two is more sensitive to the
simulation length (5 days here)? Why?

*Your answer:*

## 6. A large step: $h$ = 12 h

Repeat steps 3–4 with $h$ = 12 h (store τ in `tau_est_12h`) and plot both responses. What happens
to the τ estimate, and why? Hint: `euler_amplification` in the session notebook, $g = 1 - h/τ$.

In [ ]:
# TODO: simulate with h = 12 h, estimate tau (tau_est_12h), plot both responses

*Your answer:*

## Extension: τ from noisy data

Add Gaussian noise (σ = 0.1 K) to the simulated temperature (10-min samples) and estimate τ from
a linear fit of $\ln|T - T_{ss}|$ against $t$ (slope $-1/τ$) over the first $2τ$. Use a fixed
random seed. How much data do you need for 5 % accuracy? Try shorter fit windows.

In [ ]:
# TODO (optional)

*Your answer:*